#### Управляющий кофейни заметил, что по будням `Afternoon` (14:00–17:00) - "мертвый час". 

Проверим гипотезу: действительно ли выручка в Afternoon ниже, чем в Evening? И если да - насколько она отстает от потенциала?  

In [ ]:
# Настройка путей 
import sys
from pathlib import Path

# Получаем путь к корню проекта: поднимаемся из "Jupyter Notebooks/" на уровень выше
project_root = Path().resolve().parent

# Добавляем корень проекта в sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root added to sys.path: {project_root}")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from config import RAW_DATA_DIR
from src.db.queries import run_query

# Загрузка данных
df = pd.read_csv(RAW_DATA_DIR / "third_wave_coffee_shop.csv", parse_dates=['datetime', 'sale_date'])


Вначале я начал с готовой колонки `time_of_day`. Но, зная работу кофейни, заподозрил: `Afternoon с 11 до 17 часов` — слишком грубый срез. В `12–14 часов пик обеда`: офисные сотрудники заказывают кофе с перекусом. А вот `после 14:00` — действительно **"мёртвый час"**. Поэтому я перешёл к фильтрации по `sale_time`, чтобы проверить гипотезу точно. 
 
`Lunch Peak:` **11:00–14:00**  
`Post-Lunch Slump:` **14:00–17:00**  
`Evening Peak:` **17:00–20:00**  

In [3]:
# Run query 

from src.db.queries import run_query

query = '''
WITH daily_revenue AS (
    SELECT 
        sale_date,
        time_of_day,
        COUNT(*) AS transactions,
        SUM(max_total_cost) AS revenue
    FROM (
        SELECT 
            sale_date,
            time_of_day,
            transaction_id,
            MAX(total_cost) AS max_total_cost
        FROM third_wave_coffee_shop
        WHERE is_weekend = FALSE
          AND time_of_day IN ('Afternoon', 'Evening')
        GROUP BY sale_date, time_of_day, transaction_id
    ) AS unique_checks
    GROUP BY sale_date, time_of_day
),
summary AS (
    SELECT 
        time_of_day,
        COUNT(DISTINCT sale_date) AS days,
        SUM(revenue) AS total_revenue,
        ROUND(AVG(revenue), 2) AS avg_daily_revenue
    FROM daily_revenue
    GROUP BY time_of_day
)
SELECT
	s.time_of_day,
	s.days,
	s.total_revenue,
	s.avg_daily_revenue,
	CASE
		WHEN s.time_of_day = 'Afternoon'
			THEN e.avg_daily_revenue - s.avg_daily_revenue
		ELSE NULL
	END AS gap_to_evening_rub,
	CASE
		WHEN s.time_of_day = 'Afternoon'
			THEN ROUND(100.0 * (e.avg_daily_revenue - s.avg_daily_revenue) /
e.avg_daily_revenue, 1)
		ELSE NULL
  	END AS gap_to_evening_pct
FROM summary s
LEFT JOIN summary e ON e.time_of_day = 'Evening'
ORDER BY s.time_of_day;
'''
df = run_query(query)
df


2025-12-19 18:15:56.832 | INFO     | src.db.queries:run_query:28 - Query returned 2 rows


,time_of_day,days,total_revenue,avg_daily_revenue,gap_to_evening_rub,gap_to_evening_pct
0,Afternoon,66,610350.0,9247.73,-7335.23,-383.5
1,Evening,64,122400.0,1912.50,NaN,NaN


In [4]:
# Run query 

from src.db.queries import run_query

query = '''
WITH unique_transactions AS (
    SELECT 
        sale_date,
        day_name,
        EXTRACT(ISODOW FROM sale_date::date) as day_num, 
        transaction_id,
        MAX(total_cost) as check_amount,
        CASE 
            WHEN sale_time::time BETWEEN '11:00:00' AND '13:59:59' THEN '1. Lunch (11-14)'
            WHEN sale_time::time BETWEEN '14:00:00' AND '16:59:59' THEN '2. Dead (14-17)'
            WHEN sale_time::time >= '17:00:00' THEN '3. Evening (17-20)'
        END as time_segment
    FROM third_wave_coffee_shop
    WHERE is_weekend = FALSE
      AND sale_time >= '11:00:00'
    GROUP BY 1, 2, 3, 4, sale_time
)
SELECT 
    day_name,
    day_num,
    time_segment,
    COUNT(DISTINCT transaction_id) as total_checks,
    SUM(check_amount) as total_revenue,
    ROUND(SUM(check_amount) / 3.0, 2) as revenue_per_hour
FROM unique_transactions
WHERE time_segment IS NOT NULL
GROUP BY 1, 2, 3
ORDER BY day_num, time_segment;
  
'''
df = run_query(query)
df


2025-12-19 18:15:56.877 | INFO     | src.db.queries:run_query:28 - Query returned 15 rows


,day_name,day_num,time_segment,total_checks,total_revenue,revenue_per_hour
0,Monday,1.0,1. Lunch (11-14),196,90740.0,30246.67
1,Monday,1.0,2. Dead (14-17),75,30550.0,10183.33
2,Monday,1.0,3. Evening (17-20),41,19380.0,6460.00
3,Tuesday,2.0,1. Lunch (11-14),190,86920.0,28973.33
4,Tuesday,2.0,2. Dead (14-17),76,36700.0,12233.33
5,Tuesday,2.0,3. Evening (17-20),50,25010.0,8336.67
6,Wednesday,3.0,1. Lunch (11-14),177,87100.0,29033.33
7,Wednesday,3.0,2. Dead (14-17),67,32210.0,10736.67
8,Wednesday,3.0,3. Evening (17-20),47,22000.0,7333.33
9,Thursday,4.0,1. Lunch (11-14),204,92250.0,30750.00


**Выводы:** 
1. Гипотеза подтвердилась частично: Интервал 14:00–17:00 действительно имеет низкую эффективность по сравнению с обедом.
2. Вечерняя смена (17:00+) работает еще менее эффективно с точки зрения потока денег в кассу, хотя каждый отдельный гость оставляет там больше всего денег.
3. Рекомендация:
    * Для Dead Hours: Нужны акции на трафик.
    * Для Evening: Нужно привлекать больше компаний, так как чек уже высокий, не хватает только количества гостей.
